In [1]:
import os
from google.colab import userdata
os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")
os.environ["GOOGLE_API_KEY"] = userdata.get("GEM_API_KEY")

In [2]:
!pip install langchain-google-genai langchain-pinecone pinecone-client

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.6 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.3
    Uninstalling packaging-26.3:
      Successfully uninstalled packaging-26.3
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49

In [2]:
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [3]:
pc = Pinecone()
index = pc.Index("fda-drug-labels")

gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    output_dimensionality=768
)
#this for embed the user input so it has the same size as the stored knowlege
vectorstore = PineconeVectorStore(index=index, embedding=gemini_embeddings)

### The Retriever phase

In [4]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

In [9]:
results = retriever.invoke("What drugs treat rheumatoid arthritis?")


In [10]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(f"Drug: {doc.metadata['drug_name']}")
    print(f"Source: {doc.metadata['source']}")
    print(f"Text: {doc.page_content[:500]}...")


--- Result 1 ---
Drug: ACTEMRA_TOCILIZUMAB
Source: ACTEMRA_TOCILIZUMAB.pdf
Text: What is ACTEMRA? 
ACTEMRA is a prescription medicine called an Interleukin-6 (IL-6) receptor antagonist. ACTEMRA is used:  
• To treat adults with moderately to severely active rheumatoid arthritis (RA), after at least one other medicine called 
a Disease-Modifying Anti-Rheumatic Drug (DMARD) has been used and did not work well.  
• To treat adults with giant cell arteritis (GCA). 
• For slowing the rate of decline in lung function in adults with systemic sclerosis-associated interstitial lung d...

--- Result 2 ---
Drug: ACTEMRA_TOCILIZUMAB
Source: ACTEMRA_TOCILIZUMAB.pdf
Text: of tocilizumab administered by the intravenous route at a dose of 50 mg/kg every three days. 
14 CLINICAL STUDIES 
14.1 Rheumatoid Arthritis – Intravenous Administration 
The efficacy and safety of intravenously administered ACTEMRA was assessed in five randomized, double-blind, 
multicenter studies in patients greater than 18 yea

In [14]:
results[4]

Document(id='b7eb8428-2924-420f-8566-d5edadc360cb', metadata={'drug_name': 'ACTEMRA_TOCILIZUMAB', 'file_type': 'Monoclonal_antibodies', 'source': 'ACTEMRA_TOCILIZUMAB.pdf'}, page_content='20 10 \n-4.2 \n(-6.1, -2.3) \n19.5 8 \n-6.2 \n(-8.1, -4.2) \n21 15 19.5 13 \n-6.2 \n(-9.0, -3.5) \n19 11 \n-7.2 \n(-9.9, -4.5) \n19 18 \nPainb 61 33 \n-11.0 \n(-17.0, -5.0) \n60 30 \n-15.8 \n(-21.7, -9.9) \n57 43 63.5 43 \n-12.4 \n(-22.1, -2.1) \n65 33 \n-23.9 \n(-33.7, -14.1) \n64 48 \nPatient \nglobal \nassessmentb \n66 34 \n-10.9 \n(-17.1, -4.8) \n65 31 \n-14.9 \n(-20.9, -8.9) \n64 45 70 46 \n-10.0 \n(-20.3, 0.3) \n70 36 \n-17.4 \n(-27.8, -7.0) \n71 51 \nPhysician \nglobal \nassessmentb \n64 26 \n-5.6 \n(-10.5, -0.8) \n64 23 \n-9.0 \n(-13.8, -4.2) \n64 32 66.5 39 \n-10.5 \n(-18.6, -2.5) \n66 28 \n-18.2 \n(-26.3, -10.0) \n67.5 43 \nDisability \nindex \n(HAQ)c \n1.64 1.01 \n-0.18 \n(-0.34, -0.02) \n1.55 0.96 \n-0.21 \n(-0.37, -0.05) \n1.55 1.21 1.67 1.39 \n-0.25 \n(-0.42, -0.09) \n1.75 1.34 \n-0.34 \

so as we can see from the tests the retriver can return results from deffrente sources or from just one source so we can accepte the reseult but we have a smale probleme wish is the table or numirical chunks so we need to solve this we can use a reranker but since we have only 4 document we can't be 100% sure of our reseults so for now we will accepte the reseults for now an go to the prompting phase


prompt

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [15]:
prompt = ChatPromptTemplate.from_template("""
You are a medical information assistant for FDA monoclonal antibody drug labels.

RULES:
- Only answer using the context below.
- If the answer isn't in the context, say so.
- Always cite drug names.
- Never give personalized medical advice.

Context:
{context}

Question: {question}

Answer:
""")

In [11]:
#this is the function that will orginise the chunck or info to the model
def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[Drug: {d.metadata.get('drug_name', 'Unknown')}]\n{d.page_content}"
        for d in docs
    )

In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=userdata.get("GEM_API_KEY"),
    temperature=0.1
)

In [20]:
"""so here the question or user input will be given to this chain and go to question and context
at the context it will go to retriver then to  format_docs and for the question as it is after this thos parametre will go to the propmt
then to the llm and then return as redebul output with the stroutputParser"""
rag_chain=({"context": retriever | format_docs, "question" : RunnablePassthrough()} | prompt | llm | StrOutputParser()
           )

In [21]:
answer = rag_chain.invoke("What drugs treat rheumatoid arthritis?")
answer

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'Based on the provided context, the drugs that treat rheumatoid arthritis are:\n\n* **ACTEMRA (tocilizumab)**\n* **ABRILADA (adalimumab-afzb)**\n\n*(Note: I cannot provide personalized medical advice.)*'

In [22]:
print(answer)

Based on the provided context, the drugs that treat rheumatoid arthritis are:

* **ACTEMRA (tocilizumab)**
* **ABRILADA (adalimumab-afzb)**

*(Note: I cannot provide personalized medical advice.)*


In [24]:
DISCLAIMER = (
    "\n\n---\n This information is for educational purposes only and is "
    "not medical advice. Always consult a qualified healthcare provider."
)

answer = rag_chain.invoke("What drugs treat rheumatoid arthritis?") + DISCLAIMER

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [25]:
print(answer)

Based on the provided context, the drugs that treat rheumatoid arthritis are:

* **ACTEMRA (tocilizumab)**
* **ABRILADA (adalimumab-afzb)** 

*(Note: Please consult a healthcare professional for personalized medical advice regarding treatments.)*

---
 This information is for educational purposes only and is not medical advice. Always consult a qualified healthcare provider.


In [26]:
answer = rag_chain.invoke("What is the dose of Actemra for a 70kg patient?") + DISCLAIMER

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [27]:
print(answer)

Based on the provided context, the dose of ACTEMRA (tocilizumab) for a patient weighing at or above 30 kg (such as a 70 kg patient) depends on the condition being treated:

* **Cytokine Release Syndrome (CRS):** 8 mg per kg (alone or in combination with corticosteroids).
* **Coronavirus Disease 2019 (COVID-19):** 8 mg per kg administered as a single 60-minute intravenous infusion (an additional infusion may be administered at least 8 hours later if clinical signs or symptoms worsen or do not improve). Doses exceeding 800 mg per infusion are not recommended.
* **Systemic Juvenile Idiopathic Arthritis (SJIA):** 8 mg per kg given once every 2 weeks as a 60-minute intravenous infusion, or 162 mg subcutaneously once every week.
* **Rheumatoid Arthritis (RA):** 
  * Intravenous: Starting dose of 4 mg per kg every 4 weeks, followed by an increase to 8 mg per kg every 4 weeks based on clinical response. Doses exceeding 800 mg per infusion are not recommended.
  * Subcutaneous (for patients at 

In [28]:
answer = rag_chain.invoke("What is today's date") + DISCLAIMER
print(answer)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided context, the answer is not in the context.

---
 This information is for educational purposes only and is not medical advice. Always consult a qualified healthcare provider.


In [30]:
rag_chain.invoke("What treats melanoma?")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'Based on the provided context, the answer to what treats melanoma is not mentioned. \n\nAccording to the provided text for the drug **ABRILADA_ADALIMUMAB-AFZB**, melanoma is listed among the most frequently observed malignancies (other than lymphoma and NMSC) in uncontrolled clinical trials of adalimumab in adult patients with certain conditions, but the context does not contain information on how melanoma is treated.'

### Session memory

In [41]:
!pip install langchain-community


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.59.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [5]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory


/tmp/ipykernel_48351/1353520745.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory


In [6]:
sessions = {}

def get_session_history(session_id):
    if session_id not in sessions:
        sessions[session_id] = ChatMessageHistory()
    return sessions[session_id]

we need to change the prompte also so it could see the previous context and the chathistory also

In [9]:
prompt = ChatPromptTemplate.from_template("""
You are a medical information assistant for FDA monoclonal antibody drug labels.

RULES:
- Only answer using the context below.
- If the answer isn't in the context, say so.
- Always cite drug names.
- Never give personalized medical advice.

Previous conversation:
{chat_history}

Context:
{context}

Question: {question}

Answer:
""")

In [ ]:
#this is the function that will orginise the chunck or info to the model
def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[Drug: {d.metadata.get('drug_name', 'Unknown')}]\n{d.page_content}"
        for d in docs
    )

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=userdata.get("GEM_API_KEY"),
    temperature=0.1
)

In [14]:
# rag_chain = (
#     {
#         "context": retriever | format_docs,
#         "question": RunnablePassthrough(),
#         "chat_history": lambda x: x.get("chat_history", [])
#     }
#     | prompt
#     | llm
#     | StrOutputParser()
# )

In [17]:
from operator import itemgetter

rag_chain = (
    {
        "context": itemgetter("question") | retriever | format_docs,
        "question": itemgetter("question"),
        "chat_history": itemgetter("chat_history")
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [18]:
conversational_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [21]:
print(conversational_chain.invoke(
    {"question": "What treats RA?"},
    config={"configurable": {"session_id": "user_1"}}
))

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided context, **ACTEMRA (tocilizumab)** is used to treat adults with moderately to severely active rheumatoid arthritis (RA), after at least one other medicine called a Disease-Modifying Anti-Rheumatic Drug (DMARD) has been used and did not work well. 

*(Note: As a medical information assistant, I cannot provide personalized medical advice. Please consult a healthcare provider for any questions regarding treatments.)*


In [22]:
print(conversational_chain.invoke(
    {"question": "What about their side effects?"},
    config={"configurable": {"session_id": "user_1"}}
))

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided context, the side effects for each drug include:

**ADBRY (tralokinumab)**
* **Serious side effects:** Allergic reactions (hypersensitivity), including a severe reaction known as anaphylaxis (with symptoms such as breathing problems, fainting, dizziness, feeling lightheaded/low blood pressure, swelling of the face, mouth, and tongue, skin rash, hives, and itching), and eye problems (such as new or worsening eye pain or changes in vision).
* **Most common side effects:** 
  * Upper respiratory tract infections
  * Eye and eyelid inflammation (including redness, swelling, and itching)
  * Injection site reactions
  * High count of a certain white blood cell (eosinophilia)

**ACTEMRA (tocilizumab)**
* **Serious / other mentioned conditions:** Nervous system problems, including Multiple Sclerosis (which has been rarely diagnosed in people who take ACTEMRA), fast heartbeat or pounding in the chest (tachycardia), and sweating.
* **Most common side effects:** 
  * Upper 

we can see that we have adbry wish is not one of the treatment of the RA in the first response and this because of the their in the search we can't tell what is the their so we found the side effecte in general so the next stepe is adding a rewrting promet so we can reforme the question and have the write reseults

Query rewriting

In [26]:
rewrite_prompt = ChatPromptTemplate.from_template("""
Given the following conversation and a follow-up question, rewrite the follow-up
question into a fully standalone question that can be understood on its own.

Rules:
- Replace all pronouns (it, they, their, etc.) with the specific subjects they refer to.
- Include all necessary context from the conversation.
- If the question is already standalone, return it unchanged.
- Return ONLY the rewritten question — no explanations, no extra text.

Conversation history:
{chat_history}

Follow-up question:
{question}

Standalone question:
""")

In [25]:
def should_rewrite(question, chat_history):

#this fuction is to see if the rewriting is needed or not if there is an umbegity for example if there is pronouns like their it ...

    if not chat_history or len(chat_history) == 0:
        return False

    # very short questions
    if len(question.split()) <= 5:
        return True

    # pronouns
    pronouns = {
        "it", "its", "they", "them", "their", "theirs",
        "this", "that", "these", "those",
        "he", "she", "him", "her", "his", "hers",
        "the same", "the other", "the former", "the latter"
    }
    question_words = set(question.lower().split())
    if question_words & pronouns:
        return True

    #follow-up phrases
    follow_up_starters = [
        "what about", "how about", "and what", "and how",
        "tell me more", "explain more", "go on",
        "also", "then", "so", "but"
    ]
    if any(question.lower().startswith(s) for s in follow_up_starters):
        return True
    return False

In [27]:
rewrite_chain = rewrite_prompt | llm | StrOutputParser()

In [28]:
def prepare_query(question, chat_history):
    if not should_rewrite(question, chat_history):
        return question

    rewritten = rewrite_chain.invoke({
        "question": question,
        "chat_history": chat_history
    })
    return rewritten.strip()

In [30]:
def prepare_query_input(input_dict):
    """return rewritten question."""
    question = input_dict["question"]
    chat_history = input_dict.get("chat_history", [])
    return prepare_query(question, chat_history)

In [31]:
rag_chain = (
    {
        "context": prepare_query_input | retriever | format_docs,
        "question": itemgetter("question"),
        "chat_history": itemgetter("chat_history"),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [32]:
conversational_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [36]:
print(conversational_chain.invoke(
    {"question": "What treats RA?"},
    config={"configurable": {"session_id": "user_1"}}
))

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided context, the following drugs are used to treat rheumatoid arthritis (RA):

* **ACTEMRA (tocilizumab)** is used to treat adults with moderately to severely active rheumatoid arthritis (RA), after at least one other medicine called a Disease-Modifying Anti-Rheumatic Drug (DMARD) has been used and did not work well.
* **ABRILADA (adalimumab-afzb)** is indicated for active rheumatoid arthritis and can be used alone or in combination with methotrexate or other non-biologic disease-modifying anti-rheumatic drugs (DMARDs).

*(Note: As a medical information assistant, I cannot provide personalized medical advice. Please consult a healthcare provider for any questions regarding treatments.)*


In [37]:
print(conversational_chain.invoke(
    {"question": "What about their side effects?"},
    config={"configurable": {"session_id": "user_1"}}
))

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided context, the side effects for **ABRILADA (adalimumab-afzb)** include:

* **Serious / other mentioned conditions:**
  * New or worsening psoriasis (red scaly patches or raised bumps filled with pus)
  * Increased risk of serious infections when used with abatacept or anakinra
  * Exacerbation or new onset of demyelinating disease
  * Cytopenias and pancytopenia
  * Worsening or new onset of heart failure
  * Lupus-like syndrome
  * Rare types of cancer, such as hepatosplenic T-cell lymphoma
* **Most common side effects (>10%):**
  * Infections (including upper respiratory infections and sinus infections)
  * Injection site reactions (redness, rash, swelling, itching, or bruising)
  * Headaches
  * Rash

*(Note: As a medical information assistant, I cannot provide personalized medical advice. Please consult a healthcare provider for any questions regarding side effects.)*
